In [1]:
import pandas as pd
import numpy as np

# Find which downstream genes are affected by mutations in intergenic regions
Result: [context, start, end, first gene] (one line per first)

In [2]:
position_to_context = pd.read_csv('../../../datasets/genomics/04_gene_attribution/intermediate/position_to_context.tsv', sep = '\t')
transcription_units = pd.read_csv('../../../datasets/genomics/external_datasets/transcription_unit_regulonDB.tsv', sep = '\t').tuGenes

In [3]:
# List of intergenic regions
ig_to_first_gene = position_to_context.iloc[np.where(position_to_context.target_type == 'intergenic')].reset_index(drop = True)

# Find first upstream and downstream genes
ig_to_first_gene['upstream'] = ig_to_first_gene.apply(lambda x: x.context.split('/')[0] if x.strand.split('/')[0] == '<' else '', axis = 1)
ig_to_first_gene['downstream'] = ig_to_first_gene.apply(lambda x: x.context.split('/')[1] if x.strand.split('/')[1] == '>' else '', axis = 1)

# Transcription units following a given gene
first_gene_to_TU = pd.DataFrame({'first_gene': [i.split(',')[0] for i in transcription_units], 'TU': transcription_units})

genes_not_in_dataset = [i for i in [i for i in ig_to_first_gene.upstream if i not in first_gene_to_TU.first_gene] + [i for i in ig_to_first_gene.downstream if i not in first_gene_to_TU.first_gene] if i]
first_gene_to_TU = pd.concat([first_gene_to_TU, pd.DataFrame({'first_gene': genes_not_in_dataset, 'TU': genes_not_in_dataset})])
first_gene_to_TU['TU'] = first_gene_to_TU.TU.apply(lambda x: x.split(','))
first_gene_to_TU[first_gene_to_TU.first_gene == 'thrL']
first_gene_to_TU = first_gene_to_TU.groupby(['first_gene']).TU.aggregate(lambda x: set.union(*map(set, x))).reset_index()

# Merge and concatenate in string form
ig_to_first_gene = ig_to_first_gene.merge(first_gene_to_TU, left_on = 'upstream', right_on = 'first_gene', how = 'left')
ig_to_first_gene = ig_to_first_gene.merge(first_gene_to_TU, left_on = 'downstream', right_on = 'first_gene', how = 'left')

ig_to_first_gene['TU_x'] = ig_to_first_gene.TU_x.apply(lambda x: ','.join(x) if not type(x) == float else '')
ig_to_first_gene['TU_y'] = ig_to_first_gene.TU_y.apply(lambda x: ','.join(x) if not type(x) == float else '')
ig_to_first_gene['affected'] = (ig_to_first_gene.TU_x + ',' + ig_to_first_gene.TU_y).str.strip(',')

ig_to_first_gene.drop(columns = ['target_type', 'upstream', 'downstream', 'first_gene_x', 'first_gene_y', 'TU_x', 'TU_y'], inplace = True)

# Save
ig_to_first_gene.to_csv('../../../datasets/genomics/05_genome_mapping/results/ig_to_first_gene.tsv', sep = '\t', index = False, encoding='utf-8-sig')
ig_to_first_gene.head(5)

,chrom,start,end,context,strand,context_part,affected
0,U00096,1,189,yjtD/thrL,>/>,1,"thrB,thrL,thrC,thrA"
1,U00096,256,336,thrL/thrA,>/>,1,thrA
2,U00096,2800,2800,thrA/thrB,>/>,1,thrB
3,U00096,5021,5233,thrC/yaaX,>/>,1,yaaX
4,U00096,5531,5682,yaaX/yaaA,>/<,1,


# Name to details
Name is either a gene name, or an intergenic region name. The same locus can be on different rows, if there are overlapping genes.

In [4]:
# Genic regions
gene_map = pd.read_csv('../../../datasets/genomics/reference_genome/gene_map.tsv', sep = '\t')
gene_map['affected'] = gene_map.gene_name
gene_map['strand'] = gene_map['strand'].map({'+':'>', '-':'<'})
gene_map['name'] = gene_map['gene_name']
gene_map.drop(columns = 'gene_name', inplace = True)
display(gene_map.head(3))

# Intergenic regions
intergenic_map = pd.DataFrame(ig_to_first_gene)
intergenic_map['name'] = intergenic_map['context']
intergenic_map.drop(columns = 'context', inplace = True)
intergenic_map['target_type'] = 'intergenic'
intergenic_map[['product', 'locus_tag', 'ecocyc', 'uniprot', 'synonyms']] = ''
display(intergenic_map.head(3))

name_to_details = pd.concat([gene_map, intergenic_map])
name_to_details = name_to_details[name_to_details.chrom == 'U00096']
name_to_details['footprint'] = name_to_details.end - name_to_details.start + 1
name_to_details = name_to_details[['name', 'chrom', 'start', 'end', 'footprint', 'strand', 'target_type', 'product', 'affected', 'locus_tag', 'synonyms']]
name_to_details.sort_values(['chrom', 'start', 'end', 'name'], inplace = True)
name_to_details.reset_index(drop = True)
name_to_details.to_csv('../../../datasets/genomics/05_genome_mapping/results/name_to_details.tsv', sep = '\t', index = False)

,chrom,start,end,strand,target_type,product,locus_tag,ecocyc,uniprot,synonyms,affected,name
0,U00096,190,255,>,protein_coding,thr operon leader peptide,b0001,EG11277,P0AD86,ECK0001,thrL,thrL
1,U00096,337,2799,>,protein_coding,fused aspartate kinase/homoserine dehydrogenase 1,b0002,EG10998,P00561,"ECK0002,Hs,thrA1,thrA2,thrD",thrA,thrA
2,U00096,2801,3733,>,protein_coding,homoserine kinase,b0003,EG10999,P00547,ECK0003,thrB,thrB


,chrom,start,end,strand,context_part,affected,name,target_type,product,locus_tag,ecocyc,uniprot,synonyms
0,U00096,1,189,>/>,1,"thrB,thrL,thrC,thrA",yjtD/thrL,intergenic,,,,,
1,U00096,256,336,>/>,1,thrA,thrL/thrA,intergenic,,,,,
2,U00096,2800,2800,>/>,1,thrB,thrA/thrB,intergenic,,,,,


# Add affected genes
For intergenic genes, all genes in the operons downstream are affected.

In [5]:
d_labeled = pd.read_csv('../../../datasets/genomics/04_gene_attribution/results/d_labeled.tsv', sep = '\t', keep_default_na = False)

d_aff = d_labeled.copy()
d_aff['intergenic'] = d_aff.context.str.contains('/')

# Add affected genes
d_aff = d_aff.merge(ig_to_first_gene[['context', 'affected']], how = 'left')
d_aff['affected'] = np.where(d_aff.intergenic, d_aff.affected, d_aff.context)

d_aff.to_csv('../../../datasets/genomics/05_genome_mapping/results/d_aff.tsv', sep = '\t', index = False)